# Comparação de modelos candidatos

Notebook integrado da comparação da Seção 4.4 (ART.7). Ele reúne, em um só lugar, todos os modelos
candidatos à predição de Detrator, treinados sobre **a mesma matriz** e avaliados com
**as mesmas métricas e a mesma partição**, para que a equipe possa comparar os resultados de maneira
organizada.

A Seção 5 treina e registra os seis modelos de `scripts/comparar_seis_modelos.py`: cinco candidatos
(Extra Trees, Regressão Logística, Árvore de Decisão, Random Forest e Gradient Boosting) e a Classe
Majoritária, que é um piso de referência, e não um candidato. A Seção 6 mede todos os modelos na
partição de validação, e a Seção 6.1 calibra a probabilidade do Gradient Boosting. A Seção 7 compara
os resultados com as metas de negócio, e a Seção 8 registra a escolha do grupo. As partes comuns
(carga, pré-processamento, avaliação e visualização) também funcionam sem modelos treinados.

## Como executar

```bash
pip install -r requirements.txt
jupyter lab notebooks/comparacao_modelos.ipynb
```

Para a execução verificável de ponta a ponta, rode a partir da raiz do repositório:

```bash
jupyter nbconvert --execute --to notebook --output-dir=.execucao notebooks/comparacao_modelos.ipynb
```

**Pré-requisito**: a base `data/processed/base_analitica.parquet`, gerada por
`notebooks/pre-processamento.ipynb`. Ela não é versionada, porque o Termo de Abertura impede a
publicação de dados do parceiro.

**Sem acesso à base real**: gere a base sintética e execute o notebook com `USAR_BASE_DUMMY=1`. A
base sintética é usada apenas para verificar a execução, não para comparar modelos.

```bash
python scripts/gerar_dummy.py
USAR_BASE_DUMMY=1 jupyter nbconvert --execute --to notebook --output-dir=.execucao notebooks/comparacao_modelos.ipynb
```

Para abrir o notebook no Jupyter com a base sintética, inicie o servidor com
`USAR_BASE_DUMMY=1 jupyter lab notebooks/comparacao_modelos.ipynb`.

**Como adicionar um candidato**: crie uma subseção na Seção 5 com justificativa e parâmetros. Em
seguida, instancie o estimador, ajuste-o somente com `X_treino` e `y_treino` e registre o modelo
treinado em `modelos` com um nome único. Por fim, acrescente o modelo à tabela do início da Seção 5.
As Seções 6 e 7 o incluem automaticamente, e a Seção 5.1 serve de exemplo de instanciação, ajuste e
registro.

Rode as células **em ordem**, de cima para baixo.

## 1. Importações

Esta seção importa as bibliotecas do notebook, os estimadores do scikit-learn usados na Seção 5 e,
de `src/`, a função `preparar_matriz` (Seção 3), a função `metricas_no_teste` e a constante
`METRICA_PRINCIPAL` (Seção 6), além das funções `calibrar_se_necessario` e `brier_antes_e_depois`
(Seção 6.1).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

try:
    import sklearn
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import (
        ExtraTreesClassifier,
        HistGradientBoostingClassifier,
        RandomForestClassifier,
    )
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier
except ImportError as erro:
    raise ImportError(
        "Instale as dependências com pip install -r requirements.txt para usar os modelos."
    ) from erro

# Inclui src/ no caminho de importação, tanto no projeto local quanto no Colab.
for pasta_codigo in (Path("src"), Path("../src"), Path("/content/src")):
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from calibracao import brier_antes_e_depois, calibrar_se_necessario
from matriz import preparar_matriz
from modelo import METRICA_PRINCIPAL, metricas_no_teste

print(f"scikit-learn {sklearn.__version__}")

## 2. Carregamento dos dados

A variável de ambiente `USAR_BASE_DUMMY` seleciona a base. Sem ela, ou com qualquer valor diferente
de `1`, a célula carrega a base analítica real, que é a única que mede o desempenho dos modelos. Com
`USAR_BASE_DUMMY=1`, a célula carrega a base sintética de `scripts/gerar_dummy.py`. Ela serve apenas
para verificar se o notebook funciona sem o dado do parceiro, e suas métricas não valem para a
comparação.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount("/content/drive")

USAR_BASE_DUMMY = os.environ.get("USAR_BASE_DUMMY", "0") == "1"

if USAR_BASE_DUMMY:
    CAMINHOS_POSSIVEIS = [
        Path("data/dummy/base_analitica_dummy.parquet"),
        Path("../data/dummy/base_analitica_dummy.parquet"),
    ]
    INSTRUCAO = "Gere a base com: python scripts/gerar_dummy.py"
else:
    CAMINHOS_POSSIVEIS = [
        Path("data/processed/base_analitica.parquet"),
        Path("../data/processed/base_analitica.parquet"),
        Path("/content/data/processed/base_analitica.parquet"),
        Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
    ]
    INSTRUCAO = (
        "Rode notebooks/pre-processamento.ipynb antes. Para só verificar que o notebook "
        "roda, sem o dado do parceiro, defina USAR_BASE_DUMMY=1 no ambiente."
    )

caminho_base = next((caminho for caminho in CAMINHOS_POSSIVEIS if caminho.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(f"{CAMINHOS_POSSIVEIS[0].name} não encontrada. {INSTRUCAO}")

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas x {df.shape[1]} colunas")
if USAR_BASE_DUMMY:
    print("ATENÇÃO: base sintética. As métricas deste notebook não valem para a comparação.")

## 3. Pré-processamento

A função `preparar_matriz`, de `src/matriz.py`, particiona a base por data e por Cliente, seleciona
as features e ajusta o pré-processador **apenas no treino**. Ela é a única fonte de `X` e `y` do
notebook: todos os modelos recebem exatamente a mesma matriz. As datas de corte vêm do registro de
decisão do #127, e `SEMENTE = 42` é a semente dos modelos da Seção 5 que têm componente aleatório.

In [ ]:
SEMENTE = 42                    # semente dos modelos da Seção 5 com componente aleatório
CORTE_VALIDACAO = "2025-07-01"  # datas de corte do registro de decisão do #127
CORTE_TESTE = "2026-01-01"

preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)

X_treino, y_treino = preparo["matrizes"]["treino"], preparo["y"]["treino"]
X_validacao, y_validacao = preparo["matrizes"]["validacao"], preparo["y"]["validacao"]

for particao in ("treino", "validacao", "teste"):
    linhas, colunas = preparo["matrizes"][particao].shape
    prevalencia = preparo["y"][particao].mean()
    print(f"{particao}: {linhas:,} linhas x {colunas} colunas | "
          f"prevalência de Detrator {prevalencia:.4f}")

## 4. Registro dos modelos

O dicionário `modelos` recebe apenas modelos já treinados com a matriz de treino: os cinco
candidatos e o piso de referência da Seção 5.6. A chave de cada modelo é o nome exibido nas Seções 6
e 7. Enquanto `modelos` estiver vazio, as Seções 6 e 7 são executadas sem comparar resultados.

In [ ]:
# Nome exibido nas Seções 6 e 7 -> estimador já ajustado com X_treino e y_treino.
modelos = {}

## 5. Modelos candidatos e piso de referência

Cada modelo recebe uma subseção com descrição, justificativa, parâmetros, treinamento e registro em
`modelos`. O nome registrado é o que aparece nas tabelas e nos gráficos das Seções 6 e 7:

| Subseção | Nome em `modelos` | Papel | Origem dos hiperparâmetros |
| --- | --- | --- | --- |
| 5.1 | `Extra Trees` | candidato | configuração inicial, sem busca |
| 5.2 | `Regressão Logística` | candidato | busca em grade do card #210 |
| 5.3 | `Árvore de Decisão` | candidato | busca em grade dos cards #231 e #232 |
| 5.4 | `Random Forest` | candidato | padrão do scikit-learn, sem busca |
| 5.5 | `Gradient Boosting` | candidato | busca aleatória do card #190 |
| 5.6 | `Classe Majoritária` | piso de referência, não candidato | regra fixa, sem busca |

Os seis modelos e seus hiperparâmetros são os de `scripts/comparar_seis_modelos.py`. O script ajusta
a Regressão Logística, a Árvore de Decisão, o Random Forest e o Gradient Boosting como `Pipeline`
sobre as colunas cruas (`preparo["x"]`); aqui, cada estimador recebe `X_treino`. O ajuste é o mesmo,
porque o `Pipeline` usa um clone do pré-processador da Seção 3, ajustado sobre as mesmas linhas de
treino, e entrega ao estimador a mesma matriz. A única diferença de parâmetro é o `n_jobs` do Extra
Trees: `-1` aqui e `1` no script. As árvores são as mesmas; mudam apenas o tempo de ajuste e, no
máximo, o último bit de `predict_proba` (Seção 5.4).

A avaliação não é a do script. O script mede, com `avaliar`, F2 e Sensibilidade no limiar por
capacidade, além de Precisão Média e ROC-AUC, na validação e no teste. Aqui, a comparação usa apenas
as três métricas da Seção 6 (`precisao_media`, `roc_auc` e `brier`), que não dependem de limiar, e
somente na partição de validação. Os valores de F2 citados nas subseções são o critério de cada
busca, medido nos folds de validação cruzada dentro do treino, e não o desempenho comparado.

### 5.1. Extra Trees

O Extra Trees permite verificar, na mesma matriz de variáveis numéricas e categóricas codificadas,
interações e relações não lineares que um modelo linear pode não capturar. A aleatorização dos
limiares de divisão oferece uma alternativa aos outros ensembles na comparação; isso é uma hipótese
de modelagem, e não um resultado de desempenho.

Abaixo estão os hiperparâmetros de uma configuração inicial reproduzível, sem busca: 300 árvores,
`max_features='sqrt'`, `min_samples_leaf=5`, `random_state=SEMENTE` (42) e `n_jobs=-1`. O peso
`balanced` é calculado exclusivamente a partir de `y_treino`, sem reamostragem e sem usar a
validação ou o teste no ajuste. Não se presume que ele traga ganho para a classe minoritária.

In [ ]:
# Configuração inicial, sem busca. O ajuste usa só X_treino, já transformada na Seção 3.
modelo_extra_trees = ExtraTreesClassifier(
    n_estimators=300,
    max_features="sqrt",
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=SEMENTE,
    n_jobs=-1,
)
modelo_extra_trees.fit(X_treino, y_treino)
modelos["Extra Trees"] = modelo_extra_trees

### 5.2. Regressão Logística

A Regressão Logística é o candidato interpretável da comparação: ela modela o logito da
probabilidade de detração como uma combinação linear das colunas da matriz, e cada coeficiente
exponenciado é um odds ratio, legível sem técnicas auxiliares. Ela também é a referência linear dos
ensembles, porque não representa interação entre variáveis: o quanto os demais candidatos a superam
mede o que a interação vale nesta base.

Os hiperparâmetros abaixo não são uma configuração inicial: são os vencedores da busca em grade do
card #210, versionados em `assets/hiperparametros_logistica.json`. A busca percorreu 8 combinações
em 5 folds `GroupKFold` por `ID_GOLDENRECORD`, ordenadas por F2, e a vencedora obteve F2 médio de
0,5189, com desvio de 0,0035 entre os folds. O `class_weight` foi o único eixo que moveu a métrica.

Dois parâmetros exigem leitura explícita:

- `l1_ratio=1.0` com `solver='liblinear'` é a **penalidade L1**. O argumento `penalty` está
  depreciado desde o scikit-learn 1.8, e `l1_ratio` passou a expressá-lo: `0.0` é L2 e `1.0` é L1.
- `max_iter=1600` vem de uma medição do card #206, e não de uma estimativa: nesta matriz, o ajuste é
  truncado nas 100 iterações do padrão da biblioteca, e coeficiente truncado é coeficiente
  provisório.

O peso `balanced` é calculado exclusivamente a partir de `y_treino`, sem reamostragem e sem usar a
validação ou o teste no ajuste. Ele foi escolhido por melhorar o F2, que depende de limiar, ao passo
que as três métricas da Seção 6 não dependem. Como a reponderação afasta a probabilidade predita da
prevalência real, ela piora o escore de Brier sem piorar a ordenação. Por isso, nesta linha, um
Brier alto indica calibração deslocada, e não capacidade de ordenar pior.

In [ ]:
# Vencedores da busca do card #210 (assets/hiperparametros_logistica.json).
# O ajuste usa só X_treino, já transformada na Seção 3.
modelo_logistica = LogisticRegression(
    C=1.0,
    l1_ratio=1.0,             # 1.0 é a penalidade L1; `penalty` está depreciado desde a 1.8
    solver="liblinear",
    class_weight="balanced",
    max_iter=1600,            # medido no card #206: as 100 iterações do padrão truncam o ajuste
    random_state=SEMENTE,
)
modelo_logistica.fit(X_treino, y_treino)
modelos["Regressão Logística"] = modelo_logistica

### 5.3. Árvore de Decisão

A Árvore de Decisão é o segundo candidato interpretável da comparação, ao lado da Regressão
Logística. Em vez de coeficientes, ela é lida como regras de condição, previsão e proporção de
Detrator na folha, extraídas e interpretadas nos cards #233 e #234. Diferentemente da Regressão
Logística, cada divisão depende das divisões acima dela; portanto, a árvore representa interação
entre variáveis por construção.

Os hiperparâmetros abaixo não são uma configuração inicial: são os vencedores da busca em grade dos
cards #231 e #232, versionados em `src/hiperparametros_arvore.json`. A busca percorreu as 48
combinações da grade do card #228 em 5 folds `GroupKFold` por `ID_GOLDENRECORD`, ordenadas por F2, e
a vencedora obteve F2 médio de 0,5078 nos folds. Na base real, a árvore vencedora tem 32 folhas.

Três parâmetros exigem leitura explícita:

- `max_depth=5` e `min_samples_leaf=50` também são requisitos de explicabilidade, e não apenas de
  ajuste: a grade limitou a profundidade a no máximo 6 e a folha a no mínimo 50 respostas, para que
  a árvore final coubesse em uma leitura de regras.
- `random_state=SEMENTE` fixa o desempate entre divisões com o mesmo ganho. Sem ele, dois ajustes
  podem escolher colunas diferentes em um empate e produzir regras distintas.

O peso `balanced` é calculado exclusivamente a partir de `y_treino`, sem reamostragem e sem usar a
validação ou o teste no ajuste. Assim como na Regressão Logística, a reponderação afasta a
probabilidade predita da prevalência real. Por isso, nesta linha, um Brier alto indica calibração
deslocada, mas não necessariamente ordenação pior.

In [ ]:
# Vencedores da busca dos cards #231 e #232 (src/hiperparametros_arvore.json).
# O ajuste usa só X_treino, já transformada na Seção 3.
modelo_arvore = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=5,              # teto também de explicabilidade: a árvore precisa virar regras
    min_samples_leaf=50,      # piso da grade do card #228; folha menor descreve ruído
    class_weight="balanced",
    random_state=SEMENTE,     # fixa o desempate entre divisões com o mesmo ganho
)
modelo_arvore.fit(X_treino, y_treino)
modelos["Árvore de Decisão"] = modelo_arvore

### 5.4. Random Forest

O Random Forest é um ensemble de árvores ajustadas de maneira independente. Diferentemente do Extra
Trees (Seção 5.1), cada árvore é ajustada sobre uma amostra bootstrap do treino e, entre as colunas
sorteadas para cada divisão, escolhe o melhor limiar, em vez de sorteá-lo. A média das árvores reduz
a variância de uma árvore isolada e preserva a capacidade de representar interação entre variáveis;
o custo é a interpretabilidade, que deixa de ser intrínseca.

Os hiperparâmetros abaixo **não** vêm de busca: são os padrões do scikit-learn 1.9.1, que constituem
a linha de base do card #187. O código declara os quatro hiperparâmetros que a documentação registra
para essa linha de base: 100 árvores (`n_estimators=100`), profundidade livre (`max_depth=None`),
`max_features='sqrt'` e nenhuma reponderação (`class_weight=None`). Os demais seguem o padrão da
biblioteca. A busca aleatória do card #189 ainda não tem vencedor versionado em
`assets/hiperparametros_random_forest.json`, e a Seção 4.4.4 da documentação registra os vencedores
como pendentes da execução. Portanto, a Seção 6 mede a linha de base do Random Forest, e não um
Random Forest otimizado.

`n_jobs=1` segue `ensembles.criar_pipeline_random_forest`. Com várias threads, as árvores são as
mesmas, mas `predict_proba` soma os votos na ordem em que as threads terminam, e essa ordem pode
alterar o último bit da probabilidade. Em um empate exato no limiar, o rótulo previsto pode mudar.
Com uma thread, o resultado se repete a cada execução, ao custo de um ajuste mais lento.

In [ ]:
# Padrões do scikit-learn 1.9.1 (linha de base do card #187), sem busca.
# O ajuste usa só X_treino, já transformada na Seção 3.
modelo_random_forest = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    max_features="sqrt",
    class_weight=None,
    random_state=SEMENTE,
    n_jobs=1,                 # uma thread: predict_proba reproduzível até o último bit
)
modelo_random_forest.fit(X_treino, y_treino)
modelos["Random Forest"] = modelo_random_forest

### 5.5. Gradient Boosting

O Gradient Boosting é o ensemble de árvores por boosting: em vez de agregar árvores independentes,
como o Extra Trees e o Random Forest, ele ajusta árvores em sequência, cada uma corrigindo o erro
acumulado pelas anteriores. Ele também representa interação entre variáveis por construção. A
implementação é o `HistGradientBoostingClassifier` do scikit-learn, escolhido no card #185 em vez do
`XGBClassifier` por não acrescentar dependência ao projeto.

Os hiperparâmetros abaixo não são uma configuração inicial: são os vencedores da busca aleatória do
card #190, versionados em `assets/hiperparametros_gradient_boosting.json`, o arquivo lido por
`ensembles.melhor_gradient_boosting()`. A busca sorteou 40 combinações do espaço do card #186
(`random_state=42`) e as avaliou em 5 folds `GroupKFold` por `ID_GOLDENRECORD`, ordenadas por F2. A
vencedora obteve F2 médio de 0,5318, com desvio de 0,0035 entre os folds. O `class_weight` foi o
eixo que separou as combinações: com `balanced`, o F2 médio ficou entre 0,512 e 0,532; sem
reponderação, entre 0,311 e 0,330.

Três parâmetros exigem leitura explícita:

- `learning_rate` e `l2_regularization` estão escritos com todas as casas decimais do JSON. Se
  fossem arredondados, produziriam outro ensemble, diferente do vencedor registrado.
- `early_stopping=False` não vem da busca: é fixado por `ensembles.criar_pipeline_gradient_boosting`
  (card #188). No padrão `'auto'`, a biblioteca separaria uma parte aleatória do treino para a
  parada antecipada, e essa parte ignoraria o agrupamento por `ID_GOLDENRECORD`.

Este modelo não é o primeiro candidato da Seção 4.3.2 da documentação (card #103). Aquele modelo
também é um `HistGradientBoostingClassifier`, mas usa `modelo.HIPERPARAMETROS_CANDIDATO`, sem
reponderação, e não faz parte deste notebook.

O peso `balanced` é calculado exclusivamente a partir de `y_treino`, sem reamostragem e sem usar a
validação ou o teste no ajuste. Assim como na Regressão Logística, `balanced` venceu por melhorar o
F2, que depende de limiar. Na validação, a Precisão Média e o ROC-AUC do vencedor ficaram
praticamente iguais aos da linha de base do card #188 (Seção 11.4 de `notebooks/ensembles.ipynb`). A
reponderação afasta a probabilidade predita da prevalência real; por isso, nesta linha, um Brier
alto indica calibração deslocada, e não ordenação pior. A Seção 6.1 corrige esse deslocamento com a
calibração de Platt, sem mudar a ordem dos scores.

In [ ]:
# Vencedores da busca do card #190 (assets/hiperparametros_gradient_boosting.json).
# O ajuste usa só X_treino, já transformada na Seção 3.
modelo_gradient_boosting = HistGradientBoostingClassifier(
    learning_rate=0.049833191601257244,   # todas as casas do JSON; arredondar muda o ensemble
    max_iter=200,
    max_leaf_nodes=61,
    min_samples_leaf=150,
    l2_regularization=1.774425485152653,  # todas as casas do JSON
    class_weight="balanced",
    early_stopping=False,     # fixado no card #188: a parte aleatória ignoraria o Cliente
    random_state=SEMENTE,     # 42, o mesmo random_state registrado no JSON
)
modelo_gradient_boosting.fit(X_treino, y_treino)
modelos["Gradient Boosting"] = modelo_gradient_boosting

### 5.6. Classe Majoritária (piso de referência)

A Classe Majoritária **não é um candidato** e não pode ser escolhida na Seção 8. Ela é o piso contra
o qual os cinco candidatos são lidos: um `DummyClassifier(strategy='most_frequent')`, a mesma regra
da Seção 2.2 de `notebooks/modelagem.ipynb`, que prevê sempre a classe mais frequente de `y_treino`,
a de não Detrator, sem usar nenhuma coluna da matriz.

Como a probabilidade de Detrator emitida pelo `DummyClassifier` é 0 em todas as linhas, os três
valores da Classe Majoritária na Seção 6 são conhecidos antes da execução:

- `precisao_media` é igual à prevalência de Detrator na validação, porque um score constante não
  ordena ninguém.
- `roc_auc` é igual a 0,5, o valor de quem ordena ao acaso.
- `brier` também é igual à prevalência de Detrator na validação, porque cada Detrator gera erro
  quadrático 1: o modelo afirma com certeza que ninguém é Detrator.

Um candidato que não supere este piso em `precisao_media` e `roc_auc` não ordena os Clientes melhor
do que o acaso. No `brier`, este piso não é a régua de calibração: uma probabilidade constante igual
à prevalência p erra menos, com Brier p × (1 − p).

O classificador não tem hiperparâmetros a buscar nem usa semente: a estratégia `most_frequent` é
determinística.

In [ ]:
# Piso de referência, não candidato: prevê sempre a classe mais frequente de y_treino.
# O fit recebe X_treino só pela interface do scikit-learn; a regra não lê nenhuma coluna.
NOME_PISO = "Classe Majoritária"
modelo_classe_majoritaria = DummyClassifier(strategy="most_frequent")
modelo_classe_majoritaria.fit(X_treino, y_treino)
modelos[NOME_PISO] = modelo_classe_majoritaria

Os modelos registrados foram ajustados apenas com a partição de treino. A Seção 6 os mede na
partição de validação, enquanto a partição de teste permanece intocada até a escolha final do modelo
e do limiar.

## 6. Avaliação

Todos os modelos registrados, os cinco candidatos e o piso de referência, são medidos na partição de
**validação** com as três métricas do #105: Precisão Média (`precisao_media`), ROC-AUC (`roc_auc`) e
escore de Brier (`brier`). Apesar do nome, `metricas_no_teste` mede a partição que recebe como
parâmetro; o teste fica reservado para o modelo final. A tabela é ordenada pela Precisão Média, a
métrica principal do projeto (`METRICA_PRINCIPAL`), do maior para o menor valor.

In [ ]:
# Apesar do nome, `metricas_no_teste` mede a partição recebida: aqui, a validação.
if modelos:
    resultados = metricas_no_teste(modelos, X_validacao, y_validacao).sort_values(
        METRICA_PRINCIPAL, ascending=False
    )
    display(resultados)
else:
    resultados = pd.DataFrame()
    print("Nenhum modelo registrado em `modelos`.")

### 6.1. Calibração do Gradient Boosting

O Gradient Boosting é o modelo que monta a fila (Seção 8), mas o `class_weight='balanced'` empurra a
probabilidade dele para cima: na validação, o score médio é 0,419, com prevalência de Detrator de
0,216. A calibração de Platt (`metodo='sigmoid'`) corrige isso com uma função logística aplicada ao
score do modelo já treinado. A função `calibrar_se_necessario`, dos cards #245 e #246, congela o
modelo com `FrozenEstimator` e ajusta apenas o mapa de calibração, com `X_treino` e `y_treino`. A
medição é feita na partição de validação, que nem o modelo nem o mapa viram.

Como a função logística é monotônica, a ordem dos Clientes não muda: a fila, a Precisão Média e o
ROC-AUC continuam os mesmos, e só o Brier muda. A calibração isotônica não é usada, pois é uma
função em degraus que pode criar empates entre scores e mudar a ordem da fila.

A tabela compara o Gradient Boosting antes e depois da calibração com um chute constante na
prevalência do treino (`DummyClassifier(strategy='prior')`). Na base real, o Brier cai de 0,183 para
0,140, abaixo do chute (0,169), e o score médio passa de 0,419 para 0,189. A execução é interrompida
se o Brier não melhorar, conforme exige o card #246.

In [ ]:
# Platt (sigmoid) sobre o Gradient Boosting já ajustado: o modelo fica congelado
# (FrozenEstimator) e só o mapa de calibração é ajustado, então a ordem da fila não muda.
modelo_gb_calibrado = calibrar_se_necessario(
    modelo_gradient_boosting, X_treino, y_treino, calibrar=True, metodo="sigmoid"
)
# Referência: o chute constante na prevalência do treino, igual para todas as linhas.
chute_na_prevalencia = DummyClassifier(strategy="prior").fit(X_treino, y_treino)

display(metricas_no_teste(
    {
        "Gradient Boosting": modelo_gradient_boosting,
        "Gradient Boosting calibrado": modelo_gb_calibrado,
        "chute na prevalência do treino": chute_na_prevalencia,
    },
    X_validacao,
    y_validacao,
))

score_sem_calibracao = modelo_gradient_boosting.predict_proba(X_validacao)[:, 1]
score_calibrado = modelo_gb_calibrado.predict_proba(X_validacao)[:, 1]

# Card #246: a calibração só é aceita se o Brier melhorar na validação.
comparacao_brier = brier_antes_e_depois(y_validacao, score_sem_calibracao, score_calibrado)
assert comparacao_brier["melhorou"], comparacao_brier
print(f"score médio na validação: {score_sem_calibracao.mean():.4f} sem calibração | "
      f"{score_calibrado.mean():.4f} calibrado | prevalência de Detrator {y_validacao.mean():.4f}")

## 7. Visualização

O gráfico tem um painel por métrica da Seção 6, com os modelos em ordem decrescente de Precisão
Média, a mesma da tabela, e o valor de cada um na ponta da barra. A linha tracejada indica a meta de
negócio da Seção 4.1.3 da documentação: Precisão Média ≥ 0,40 e ROC-AUC ≥ 0,75. O Brier não tem meta
de negócio. A meta de Sensibilidade (≥ 0,70) depende do limiar de decisão; por isso, não aparece
neste gráfico, que mostra apenas métricas independentes de limiar. O painel do Brier mostra os
modelos sem calibração, enquanto o Gradient Boosting calibrado está na Seção 6.1.

A primeira célula define as metas, as cores e a função que desenha cada painel; a segunda monta a
figura.

In [ ]:
# Nome, sentido de leitura e meta de negócio de cada métrica da Seção 6. As metas são
# as da Seção 4.1.3 da documentação; o Brier não tem meta de negócio.
METRICAS = {
    "precisao_media": ("Precisão Média", "maior é melhor", 0.40),
    "roc_auc": ("ROC-AUC", "maior é melhor", 0.75),
    "brier": ("Brier", "menor é melhor", None),
}
COR_CANDIDATO, COR_PISO = "#2a78d6", "#898781"
SUPERFICIE, GRADE, LINHA_DE_BASE = "#fcfcfb", "#e1e0d9", "#c3c2b7"
TINTA, TINTA_SECUNDARIA = "#0b0b0b", "#52514e"


def desenhar_painel(eixo, valores, cores, nome, sentido, meta):
    """Desenha uma métrica em barras horizontais, com o valor na ponta e a meta, se houver."""
    barras = eixo.barh(valores.index, valores, height=0.5, color=cores)
    # O fundo do rótulo interrompe a linha da meta só onde ela cruzaria o número.
    eixo.bar_label(barras, fmt="%.3f", padding=3, fontsize=9, color=TINTA_SECUNDARIA,
                   bbox={"facecolor": SUPERFICIE, "edgecolor": "none", "pad": 1})
    if meta is not None:
        eixo.axvline(meta, color=TINTA, linestyle="--", linewidth=1.2, zorder=2.5)
    # Eixo a partir de zero: o comprimento da barra é o valor. A folga à direita
    # reserva espaço para a meta e para o rótulo da barra mais longa.
    eixo.set_xlim(0, max(valores.max(), meta or 0) * 1.25)
    detalhe = f"meta de negócio ≥ {meta:.2f}" if meta is not None else "sem meta de negócio"
    eixo.set_title(f"{nome} ({sentido})\n{detalhe}", loc="left", fontsize=10.5, color=TINTA)
    eixo.set_facecolor(SUPERFICIE)
    eixo.grid(axis="x", color=GRADE, linewidth=0.8)
    eixo.set_axisbelow(True)
    eixo.tick_params(length=0, labelsize=10, colors=TINTA_SECUNDARIA)
    for lado in ("top", "right", "bottom"):
        eixo.spines[lado].set_visible(False)
    eixo.spines["left"].set_color(LINHA_DE_BASE)

In [ ]:
if resultados.empty:
    print("Sem resultados para visualizar.")
else:
    # `barh` desenha de baixo para cima: inverter a tabela da Seção 6 deixa no topo
    # o modelo com o maior valor da métrica principal.
    ordenados = resultados.iloc[::-1]
    cores = [COR_PISO if nome == NOME_PISO else COR_CANDIDATO for nome in ordenados.index]
    paineis = {
        coluna: METRICAS.get(coluna, (coluna, "sentido não declarado", None))
        for coluna in ordenados.columns
    }

    # Um painel por métrica; o nome do modelo fica no eixo y, sem girar nem sobrepor.
    fig, eixos = plt.subplots(
        1, len(paineis), sharey=True, squeeze=False,
        figsize=(1.6 + 3.3 * len(paineis), 1.6 + 0.42 * len(ordenados)),
        layout="constrained", facecolor=SUPERFICIE,
    )
    for eixo, (coluna, (nome, sentido, meta)) in zip(eixos[0], paineis.items()):
        desenhar_painel(eixo, ordenados[coluna], cores, nome, sentido, meta)

    legenda = [
        Patch(color=cor, label=rotulo)
        for cor, rotulo in ((COR_CANDIDATO, "candidato"),
                            (COR_PISO, "piso de referência (Seção 5.6)"))
        if cor in cores
    ]
    if any(meta is not None for _, _, meta in paineis.values()):
        legenda.append(Line2D([], [], color=TINTA, linestyle="--", linewidth=1.2,
                              label="meta de negócio (Seção 4.1.3 da documentação)"))
    metrica_principal = METRICAS.get(METRICA_PRINCIPAL, (METRICA_PRINCIPAL,))[0]
    fig.suptitle(f"Comparação dos modelos na validação, ordenados por {metrica_principal}",
                 fontsize=12, color=TINTA)
    fig.legend(handles=legenda, loc="outside lower center", ncols=len(legenda),
               frameon=False, labelcolor=TINTA_SECUNDARIA, fontsize=10)
    plt.show()

## 8. Conclusão

A decisão do grupo tem duas partes: o Gradient Boosting decide a ordem da fila de contato, e a
Regressão Logística dá o motivo.

O Gradient Boosting monta a fila porque é o modelo que melhor ordena os Clientes por risco na
partição de validação (Seção 6). Ele tem a maior Precisão Média (0,517), acima da meta de negócio de
0,40, e o maior ROC-AUC (0,733). O ROC-AUC ainda fica 0,017 abaixo da meta de 0,75, e essa diferença
deve ser tratada na próxima sprint.

Calibrado (Seção 6.1), o Gradient Boosting também passa a ter uma probabilidade mais honesta que um
chute na média. Sem calibração, o `class_weight='balanced'` empurra a probabilidade para cima, e o
Brier dele (0,183) fica pior que o de um chute constante na prevalência do treino (0,169). A
calibração de Platt leva o Brier a 0,140, abaixo tanto do chute quanto do Random Forest (0,159), que
tinha o menor Brier entre os modelos não calibrados. A fila não muda, porque a calibração preserva a
ordem dos scores: a Precisão Média e o ROC-AUC continuam os mesmos.

A Regressão Logística fica ao lado porque ordena quase tão bem quanto o Gradient Boosting, com
Precisão Média de 0,501 e ROC-AUC de 0,727, e porque cada fator vira um número que a Azul consegue
ler: o odds ratio. Segundo a Seção 4.4.2 da documentação, um cancelamento avisado no mesmo dia
multiplica por 7,73 a chance de detração em relação a um voo sem cancelamento, e o atraso na saída,
do percentil 10 ao 90 (35 minutos), multiplica essa chance por 1,986. Esses números descrevem
associações que a Regressão Logística aprendeu a partir dos mesmos dados. Eles mostram quais fatores
pesam na detração, mas não explicam a posição de cada Cliente na fila do Gradient Boosting nem
provam causa.